# Course 961731: Business Data Analytics
## Lab Assignment — Week 09A: Social Media Sentiment Analysis & Operational Triage

**Stakeholder:** VP of Customer Experience & Support Operations  
**Dataset:** CrowdFlower Twitter US Airline Sentiment (14,640 records)  
**Objective:** Build an audited, reproducible 3-class sentiment classifier and aspect triage listening pipeline.


In [ ]:
import os, re, sys, unicodedata, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score, log_loss
from sklearn.pipeline import Pipeline

sns.set_theme(style='whitegrid', palette='deep')
RANDOM_SEED = 961731
CLASS_ORDER = ['negative', 'neutral', 'positive']
CLASS_PALETTE = {'negative': '#c44e52', 'neutral': '#8c8c8c', 'positive': '#4c72b0'}
print('Environment initialized.')


## Problem 1: Ingestion, Leakage Audit & Graph Deduplication
- Preserve all 14,640 source records (=14,640$) in .
- Detect and quarantine 18 conflicting-annotation  groups (36 rows).
- Expose the  annotation leakage trap.
- Deduplicate on connected components (shared authors or clean text) to produce independent units.


In [ ]:
# 1. Ingestion
data_path = 'data/Tweets.csv'
df = pd.read_csv(data_path)
df['source_row'] = df.index
df['tweet_id'] = df['tweet_id'].astype(str)
df['tweet_created_utc'] = pd.to_datetime(df['tweet_created'], errors='coerce', utc=True)
print(f'Ingested source rows: {len(df):,}')
assert len(df) == 14640

# 2. Quarantine Conflicting Labels
conflicts = df.groupby('tweet_id')['airline_sentiment'].nunique()
conflicting_ids = set(conflicts[conflicts > 1].index)
print(f'Conflicting tweet_id groups: {len(conflicting_ids)}')
df['audit_flags'] = ''
df.loc[df['tweet_id'].isin(conflicting_ids), 'audit_flags'] = 'CONFLICTING_LABEL'
print(f'Quarantined rows: {(df["audit_flags"] == "CONFLICTING_LABEL").sum()}')

# 3. Negativereason Leakage Expose
crosstab_leakage = pd.crosstab(df['airline_sentiment'], df['negativereason'].isnull(), rownames=['Sentiment'], colnames=['is_negativereason_null'])
display(crosstab_leakage)


In [ ]:
# 4. Text Normalization
CONTRACTIONS = {
    r"can't": 'cannot', r"won't": 'will not', r"n't": ' not',
    r"'re": ' are', r"'s": ' is', r"'d": ' would',
    r"'ll": ' will', r"'t": ' not', r"'ve": ' have', r"'m": ' am'
}

def clean_text(text):
    if not isinstance(text, str):
        return ''
    norm = unicodedata.normalize('NFKC', text)
    norm = re.sub(r'https?://\S+|www\.\S+', 'urltoken', norm)
    norm = re.sub(r'@[A-Za-z0-9_]+', 'usertoken', norm)
    norm = re.sub(r'#([A-Za-z0-9_]+)', r'', norm)
    for pat, repl in CONTRACTIONS.items():
        norm = re.sub(pat, repl, norm, flags=re.IGNORECASE)
    norm = norm.lower()
    return re.sub(r'\s+', ' ', norm).strip()

df['text_clean'] = df['text'].apply(clean_text)

# Demonstrate Raw vs Clean
sample_pairs = df[['text', 'text_clean']].sample(5, random_state=RANDOM_SEED)
for i, r in sample_pairs.iterrows():
    print('RAW  :', r['text'])
    print('CLEAN:', r['text_clean'])
    print('-' * 60)


In [ ]:
# 5. Connected Components Deduplication
eligible_mask = (
    (~df['tweet_id'].isin(conflicting_ids)) &
    (df['airline_sentiment'].isin(CLASS_ORDER)) &
    (df['text_clean'] != '') &
    (df['tweet_created_utc'].notnull())
)

G = nx.Graph()
for idx, r in df[eligible_mask].iterrows():
    r_node = f'row_{idx}'
    G.add_node(r_node)
    if pd.notnull(r['name']) and str(r['name']).strip() != '':
        G.add_edge(r_node, f"author_{str(r['name']).strip()}")
    if r['text_clean'] != '':
        G.add_edge(r_node, f"text_{r['text_clean']}")

df['component_id'] = -1
df['is_representative'] = False
representatives = []
comp_id = 0
for comp in nx.connected_components(G):
    rows = [int(n.replace('row_', '')) for n in comp if n.startswith('row_')]
    if rows:
        df.loc[rows, 'component_id'] = comp_id
        sub = df.loc[rows].sort_values(['tweet_created_utc', 'tweet_id', 'source_row'])
        rep_idx = sub.index[0]
        representatives.append(rep_idx)
        df.loc[rep_idx, 'is_representative'] = True
        comp_id += 1

print(f'Total independent components: {comp_id:,}')
print(f'Selected earliest representatives: {len(representatives):,}')
df['model_eligible'] = df['is_representative'] & eligible_mask

# Stratified Split (80/20)
rep_df = df[df['model_eligible']].copy()
train_idx, test_idx = train_test_split(
    rep_df.index, test_size=0.2, random_state=RANDOM_SEED, stratify=rep_df['airline_sentiment']
)
train_df = rep_df.loc[train_idx]
test_df = rep_df.loc[test_idx]
print(f'Train units: {len(train_df):,} | Test units: {len(test_df):,}')


## Problem 2: Supervised Classification & Local Contrast
- Compare Dummy majority baseline with Sparse Multinomial Logistic Regression.
- 3-Fold Stratified Cross-Validation for  \in \{0.3, 1.0, 3.0\}$.
- Evaluate once on locked test set.
- Algebraically verify local contrast decomposition: $\Delta z = \Delta b + \sum \Delta w_i x_i$.


In [ ]:
# Cross-Validation on Training Set
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
c_grid = [0.3, 1.0, 3.0]
for c in c_grid:
    fold_f1 = []
    for tr, val in skf.split(train_df['text_clean'], train_df['airline_sentiment']):
        pipe = Pipeline([
            ('tfidf', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, smooth_idf=True, norm='l2', max_features=5000, min_df=2)),
            ('clf', LogisticRegression(solver='lbfgs', C=c, max_iter=1000, random_state=RANDOM_SEED))
        ])
        pipe.fit(train_df['text_clean'].iloc[tr], train_df['airline_sentiment'].iloc[tr])
        v_pred = pipe.predict(train_df['text_clean'].iloc[val])
        fold_f1.append(f1_score(train_df['airline_sentiment'].iloc[val], v_pred, average='macro'))
    print(f'C={c:<3} -> Mean Macro-F1: {np.mean(fold_f1):.4f}')

# Final Pipeline at C=3.0
final_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, smooth_idf=True, norm='l2', max_features=5000, min_df=2)),
    ('clf', LogisticRegression(solver='lbfgs', C=3.0, max_iter=1000, random_state=RANDOM_SEED))
])
final_pipe.fit(train_df['text_clean'], train_df['airline_sentiment'])
test_preds = final_pipe.predict(test_df['text_clean'])
test_probs = final_pipe.predict_proba(test_df['text_clean'])

print('Locked Test Accuracy:', accuracy_score(test_df['airline_sentiment'], test_preds))
print('Locked Test Macro-F1:', f1_score(test_df['airline_sentiment'], test_preds, average='macro'))
print('
Classification Report:')
print(classification_report(test_df['airline_sentiment'], test_preds, labels=CLASS_ORDER))


In [ ]:
# Local Contrast Verification
vec = final_pipe.named_steps['tfidf']
clf = final_pipe.named_steps['clf']
classes = list(clf.classes_)

sample_text = test_df['text_clean'].iloc[0]
sample_x = vec.transform([sample_text])
sample_probs = test_probs[0]
rank_order = np.argsort(sample_probs)[::-1]
win_k, run_j = rank_order[0], rank_order[1]

# Direct logit difference
logits = clf.intercept_ + clf.coef_ @ sample_x.toarray().flatten()
delta_direct = logits[win_k] - logits[run_j]

# Decomposed difference
delta_b = clf.intercept_[win_k] - clf.intercept_[run_j]
word_diffs = (clf.coef_[win_k] - clf.coef_[run_j]) * sample_x.toarray().flatten()
delta_decomposed = delta_b + np.sum(word_diffs)

print(f'Sample Text: {sample_text[:50]}...')
print(f'Winner: {classes[win_k]} | Runner-up: {classes[run_j]}')
print(f'Direct Delta z: {delta_direct:.6f}')
print(f'Decomposed    : {delta_decomposed:.6f}')
assert np.isclose(delta_direct, delta_decomposed)
print('Exact Numerical Decomposition Verified: MATCH')


## Problem 3: Aspect Triage & Visualizations
- Multi-label keyword matching across 4 operational dictionaries: , , , .
- Transparent denominator reporting ($).


In [ ]:
ASPECT_PATTERNS = {
    'delay': r'(?:delay|delayed|delays|late|cancel|cancelled|cancelling|cancellation|cancellations|holding|hours? late|on hold)',
    'baggage': r'(?:bag|bags|baggage|luggage|suitcase|suitcases|lost bag|claim)',
    'staff': r'(?:staff|crew|agent|agents|attendant|attendants|representative|representatives|flight attendant|gate agent|employee|employees|pilot|pilots)',
    'refund': r'(?:refund|refunds|refunded|refunding|voucher|vouchers|reimburse|reimbursement|compensation|fee|fees|credit|credits)'
}

for a, pat in ASPECT_PATTERNS.items():
    df[f'aspect_{a}'] = df['text'].str.contains(pat, case=False, regex=True, na=False)

df['aspect_matches'] = df[[f'aspect_{a}' for a in ASPECT_PATTERNS]].sum(axis=1)
df['aspect_no_match'] = df['aspect_matches'] == 0

summary_data = []
for a in list(ASPECT_PATTERNS.keys()) + ['no_match']:
    sub = df[df[f'aspect_{a}']]
    n_a = len(sub)
    cnts = sub['airline_sentiment'].value_counts()
    summary_data.append({
        'Aspect': a.capitalize(),
        'Matches (n_a)': n_a,
        'Negative (%)': round(cnts.get('negative', 0) / n_a * 100, 1),
        'Neutral (%)': round(cnts.get('neutral', 0) / n_a * 100, 1),
        'Positive (%)': round(cnts.get('positive', 0) / n_a * 100, 1),
    })

aspect_summary_table = pd.DataFrame(summary_data)
display(aspect_summary_table)
